# SHEBANQ controls for Hakhel


## Table of contents (ToC)<a class="anchor" id="TOC"></a>
* <a href="#bullet1">1 - Purpose of this notebook</a>
* <a href="#bullet2">2 - Install the required libraries</a>
* <a href="#bullet3">3 - Read the public query IDs</a>
* <a href="#bullet4">4 - Configure the anonymous SHEBANQ API calls</a>
* <a href="#bullet5">5 - Retrieve metadata and apply the book-name mapping</a>
* <a href="#bullet6">6 - Retrieve the predefined queries</a>
* <a href="#bullet7">7 - Export the recorded query data</a>
* <a href="#bullet8">8 - Compare SHEBANQ query 7334 with Context-Fabric</a>
* <a href="#bullet9">9 - Final checks</a>
* <a href="#bullet10">10 - Reading the control results</a>
* <a href="#bullet11">11 - Required libraries</a>
* <a href="#bullet12">12 - Input and output overview</a>
* <a href="#bullet13">13 - Responsible disclosure</a>
* <a href="#bullet14">14 - Licences</a>
* <a href="#bullet15">15 - Notebook details</a>

# 1 - Purpose of this notebook<a class="anchor" id="bullet1"></a>
##### [Back to ToC](#TOC)

This notebook checks five public SHEBANQ query records for [BHSA 2021](https://github.com/ETCBC/shebanq/wiki/Sources), covering טף and the audience terms in the Hakhel instruction.

It reads the query metadata and recorded CSV results, retains the MQL query text, and compares the all-טף word slots with the Context-Fabric export.
No SHEBANQ account is required.


# 2 - Install the required libraries<a class="anchor" id="bullet2"></a>
##### [Back to ToC](#TOC)

The notebook uses public HTTP requests and pandas tables. No SHEBANQ client credentials or browser session are required.

In [1]:
# Install the required libraries
%pip install -q "requests>=2.31" "pandas>=2.0"

Note: you may need to restart the kernel to use updated packages.


# 3 - Read the public query IDs<a class="anchor" id="bullet3"></a>
##### [Back to ToC](#TOC)

The [query catalogue](public_shebanq_queries.json) lists the five query IDs and their local descriptions.
To use a different set, update the catalogue, the export-filename mapping, and the checks that refer to these specific controls.


In [2]:
# Read the public control catalogue
from __future__ import annotations

import hashlib
import json
import re
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from IPython.display import Code, Markdown, display


NOTEBOOK_DIR = Path.cwd()
REPOSITORY_DIR = NOTEBOOK_DIR.parent
CONTEXT_FABRIC_INPUT = REPOSITORY_DIR / "context_fabric" / "results" / "taf_occurrences.csv"
QUERY_CATALOGUE = NOTEBOOK_DIR / "public_shebanq_queries.json"
OUTPUT_DIR = NOTEBOOK_DIR / "results" / "public_query_control"
QUERY_RESULT_DIR = OUTPUT_DIR / "query_results"

catalogue = json.loads(QUERY_CATALOGUE.read_text(encoding="utf-8"))
BHSA_VERSION = str(catalogue.get("bhsa_version", "")).strip()
query_specs = catalogue.get("queries") or []

assert BHSA_VERSION == "2021", "This notebook is intended for BHSA 2021."
assert query_specs, "The public control catalogue contains no query records."

print(f"Catalogue of query IDs: {QUERY_CATALOGUE.relative_to(REPOSITORY_DIR).as_posix()}")
print(f"Queries listed: {len(query_specs)}")
print(f"BHSA version: {BHSA_VERSION}")


Catalogue of query IDs: SHEBANQ-BHSA/public_shebanq_queries.json
Queries listed: 5
BHSA version: 2021


# 4 - Configure the anonymous SHEBANQ API calls<a class="anchor" id="bullet4"></a>
##### [Back to ToC](#TOC)

Two public SHEBANQ endpoints supply the data:

* `/hebrew/query.json` returns query metadata and the BHSA 2021 execution record;
* `/hebrew/item.csv` returns the recorded result rows.

The notebook makes no login call and supplies no credentials. It accepts a query only when it is publicly readable, marked as shared, and has a BHSA 2021 execution with status `good`.
Shared access and formal publication are recorded separately.


In [3]:
# Configure the anonymous SHEBANQ route (the one you use for webaccess)
BASE_URL = "https://shebanq.ancient-data.org"
REQUEST_TIMEOUT = 120
STOP_ON_ERROR = False

session = requests.Session()
session.headers.update({"User-Agent": "Hakhel SHEBANQ control notebook/1.0"})

assert session.auth is None  #check that there is no authentication

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
QUERY_RESULT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Public SHEBANQ host: {BASE_URL}")
print("Authentication used: no")
print(f"Control output: {OUTPUT_DIR.relative_to(REPOSITORY_DIR).as_posix()}")


Public SHEBANQ host: https://shebanq.ancient-data.org
Authentication used: no
Control output: SHEBANQ-BHSA/results/public_query_control


# 5 - Retrieve metadata and apply the book-name mapping<a class="anchor" id="bullet5"></a>
##### [Back to ToC](#TOC)

SHEBANQ and Context-Fabric expose the same BHSA 2021 slots, but their displayed book names differ. The `BOOK_NAME_MAP` below changes such values as `Deuteronomium` to `Deuteronomy` only for comparison. The original SHEBANQ book and reference (in Latin) are retained beside the mapped (English) reference.

In [4]:
# Retrieve query metadata and map the BHSA book names
SHORT_NAME_PATTERN = re.compile(r"^[a-z][a-z0-9_]*$")

BOOK_NAME_MAP = {
    "Numeri": "Numbers",
    "Deuteronomium": "Deuteronomy",
    "Josua": "Joshua",
    "Judices": "Judges",
    "Samuel_II": "2_Samuel",
    "Jeremia": "Jeremiah",
    "Ezechiel": "Ezekiel",
    "Esra": "Ezra",
    "Chronica_II": "2_Chronicles",
}



The next cell defines functions for validating query IDs, retrieving metadata and CSVs, mapping book names, and displaying the results.


In [5]:


def validate_query_spec(spec: dict[str, Any]) -> dict[str, Any]:
    """Validate one maintained catalogue entry."""
    if not isinstance(spec, dict):
        raise TypeError(f"Each query entry must be an object, received {spec!r}.")
    query_id = spec.get("query_id")
    short_name = str(spec.get("short_name", "")).strip()
    if isinstance(query_id, bool) or not isinstance(query_id, int) or query_id <= 0:
        raise ValueError(f"Invalid public SHEBANQ query ID: {query_id!r}")
    if not SHORT_NAME_PATTERN.fullmatch(short_name):
        raise ValueError(
            f"Invalid short_name {short_name!r}; use lower-case letters, "
            "numbers, and underscores."
        )
    return {
        "query_id": query_id,
        "short_name": short_name,
        "local_description": str(spec.get("local_description", "")).strip(),
    }


def public_query_metadata(query_id: int) -> dict[str, Any]:
    """Read one shared query record without authentication."""
    response = session.get(
        f"{BASE_URL}/hebrew/query.json",
        params={"version": BHSA_VERSION, "iid": str(query_id)},
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    try:
        payload = response.json()
    except ValueError as exc:
        raise RuntimeError(
            f"Query {query_id}: the public metadata response was not JSON."
        ) from exc
    if not payload.get("good"):
        raise RuntimeError(
            f"Query {query_id} is not publicly readable: {payload.get('msg')}"
        )
    metadata = payload.get("data") or {}
    if not metadata.get("is_shared"):
        raise RuntimeError(f"Query {query_id} is readable but is not marked as shared.")
    version_info = (metadata.get("versions") or {}).get(BHSA_VERSION)
    if not version_info:
        raise RuntimeError(
            f"Query {query_id} has no BHSA {BHSA_VERSION} execution record."
        )
    if version_info.get("status") != "good":
        raise RuntimeError(
            f"Query {query_id} has execution status {version_info.get('status')!r}."
        )
    return metadata


def public_query_csv(query_id: int, short_name: str) -> pd.DataFrame:
    """Download the recorded focus rows for one shared query."""
    response = session.get(
        f"{BASE_URL}/hebrew/item.csv",
        params={
            "version": BHSA_VERSION,
            "mr": "r",
            "qw": "q",
            "iid": str(query_id),
            "tp": "txt3",
            "extra": short_name,
        },
        timeout=REQUEST_TIMEOUT,
    )
    response.raise_for_status()
    content_type = response.headers.get("content-type", "")
    if "csv" not in content_type.lower():
        raise RuntimeError(
            f"Query {query_id}: expected CSV but received {content_type!r}."
        )
    return pd.read_csv(StringIO(response.content.decode("utf-8-sig")))


def author_name(metadata: dict[str, Any]) -> str:
    """Form the public author display name."""
    return " ".join(
        str(metadata.get(key) or "").strip()
        for key in ("first_name", "last_name")
    ).strip()


def mapped_book_name(book: Any) -> str:
    """Map only differing SHEBANQ book values to Context Fabric names."""
    value = str(book)
    return BOOK_NAME_MAP.get(value, value)


def prepare_control_table(
    frame: pd.DataFrame,
    spec: dict[str, Any],
    metadata: dict[str, Any],
    result_count: int | None,
) -> pd.DataFrame:
    """Retain source columns and add query and comparison provenance."""
    prepared = frame.copy()
    prepared.insert(0, "shebanq_query_id", spec["query_id"])
    prepared.insert(1, "shebanq_short_name", spec["short_name"])
    prepared.insert(2, "shebanq_query_name", metadata.get("name", ""))
    prepared.insert(3, "bhsa_version", BHSA_VERSION)
    prepared.insert(4, "shebanq_result_count", result_count)
    prepared.insert(5, "shebanq_csv_row_count", len(frame))
    if {"book", "chapter", "verse"}.issubset(prepared.columns):
        shebanq_reference = (
            prepared["book"].astype(str)
            + " "
            + prepared["chapter"].astype(str)
            + ":"
            + prepared["verse"].astype(str)
        )
        prepared.insert(6, "shebanq_reference", shebanq_reference)
        prepared.insert(
            7,
            "reference",
            prepared["book"].map(mapped_book_name)
            + " "
            + prepared["chapter"].astype(str)
            + ":"
            + prepared["verse"].astype(str),
        )
    else:
        prepared.insert(6, "shebanq_reference", pd.NA)
        prepared.insert(7, "reference", pd.NA)
    return prepared


def display_public_record(
    spec: dict[str, Any],
    metadata: dict[str, Any],
    version_info: dict[str, Any],
    prepared: pd.DataFrame,
) -> None:
    """Display the public record, actual MQL, and first focus rows."""
    display(Markdown(f"## {spec['short_name']} — SHEBANQ query #{spec['query_id']}"))
    compact = pd.DataFrame(
        [
            ("Public query name", metadata.get("name", "")),
            ("Local control description", spec.get("local_description", "")),
            ("Author", author_name(metadata)),
            ("BHSA version", BHSA_VERSION),
            ("Shared", metadata.get("is_shared")),
            ("Formally published", version_info.get("is_published")),
            ("Execution status", version_info.get("status")),
            ("SHEBANQ result count", version_info.get("results")),
            ("Result monads", version_info.get("resultmonads")),
            ("CSV focus rows", len(prepared)),
            ("Executed", version_info.get("executed_on")),
        ],
        columns=["Field", "Value"],
    )
    display(compact)
    display(Markdown("**Public description**"))
    display(Markdown(metadata.get("description_md") or metadata.get("description") or "_No description supplied._"))
    display(Markdown("**MQL stored with the BHSA 2021 execution**"))
    display(Code(data=version_info.get("mql") or "", language="text"))
    display(Markdown("**First rows of the control table**"))
    display(prepared.head())


# 6 - Retrieve the predefined queries<a class="anchor" id="bullet6"></a>
##### [Back to ToC](#TOC)

For each query, the notebook downloads the metadata and result rows, writes a CSV, and displays the public description and MQL returned by SHEBANQ.


In [6]:
# Retrieve the queries provided in the JSON file
validated_specs = [validate_query_spec(spec) for spec in query_specs]

query_ids = [spec["query_id"] for spec in validated_specs]
short_names = [spec["short_name"] for spec in validated_specs]
assert len(query_ids) == len(set(query_ids)), "Duplicate public query IDs."
assert len(short_names) == len(set(short_names)), "Duplicate short names."

detail_rows: list[dict[str, Any]] = []
control_frames: dict[str, pd.DataFrame] = {}

for spec in validated_specs:
    query_id = spec["query_id"]
    short_name = spec["short_name"]
    query_url = f"{BASE_URL}/hebrew/text?iid={query_id}&version={BHSA_VERSION}&mr=r&qw=q"
    try:
        metadata = public_query_metadata(query_id)
        version_info = metadata["versions"][BHSA_VERSION]
        raw_frame = public_query_csv(query_id, short_name)
        prepared = prepare_control_table(
            raw_frame, spec, metadata, version_info.get("results")
        )
        export_name = {
            7328: "7328_complete_audience.csv",
            7329: "7329_without_taf.csv",
            7333: "7333_audience_overlap.csv",
            7334: "7334_all_taf.csv",
            7335: "7335_ger_taf.csv",
        }[query_id]
        export_path = QUERY_RESULT_DIR / export_name
        prepared.to_csv(export_path, index=False, encoding="utf-8")
        control_frames[short_name] = prepared

        mql = version_info.get("mql") or ""
        detail_rows.append(
            {
                "status": "ok",
                "publicly_readable_without_login": True,
                "query_id": query_id,
                "short_name": short_name,
                "local_description": spec["local_description"],
                "query_name": metadata.get("name"),
                "public_description": metadata.get("description"),
                "author": author_name(metadata),
                "organization": metadata.get("org_name"),
                "project": metadata.get("project_name"),
                "created_on": metadata.get("created_on"),
                "modified_on": metadata.get("modified_on"),
                "shared_on": metadata.get("shared_on"),
                "bhsa_version": BHSA_VERSION,
                "is_shared": metadata.get("is_shared"),
                "is_published": version_info.get("is_published"),
                "execution_status": version_info.get("status"),
                "executed_on": version_info.get("executed_on"),
                "published_on": version_info.get("published_on"),
                "query_execution_id": version_info.get("query_exe_id"),
                "result_count": version_info.get("results"),
                "result_monads": version_info.get("resultmonads"),
                "csv_row_count": len(raw_frame),
                "mql": mql,
                "mql_sha256": hashlib.sha256(mql.encode("utf-8")).hexdigest(),
                "query_url": query_url,
                "control_csv": export_path.relative_to(REPOSITORY_DIR).as_posix(),
                "error": "",
            }
        )
        display_public_record(spec, metadata, version_info, prepared)
    except Exception as exc:
        detail_rows.append(
            {
                "status": "error",
                "publicly_readable_without_login": False,
                "query_id": query_id,
                "short_name": short_name,
                "local_description": spec["local_description"],
                "query_url": query_url,
                "error": f"{type(exc).__name__}: {exc}",
            }
        )
        display(Markdown(f"## {short_name} — SHEBANQ query #{query_id}"))
        display(Markdown(f"**Retrieval failed:** `{type(exc).__name__}: {exc}`"))
        if STOP_ON_ERROR:
            raise

query_details = pd.DataFrame(detail_rows)
query_details[[
    "query_id", "short_name", "query_name", "is_shared", "is_published",
    "result_count", "csv_row_count", "status"
]]


## all_taph — SHEBANQ query #7334

,Field,Value
0,Public query name,All occurrence of טף
1,Local control description,Control query for all 42 BHSA occurrences of ט...
2,Author,Tony Jurg
3,BHSA version,2021
4,Shared,True
5,Formally published,False
6,Execution status,good
7,SHEBANQ result count,42
8,Result monads,42
9,CSV focus rows,42


**Public description**

<p>This query forms part of a series of SHEBANQ queries investigating the Hakhel audience formula in Deuteronomy 31:12, with particular attention to טף (“little ones”).</p>
<p>This query identifies every occurrence of טף.</p>
<p><strong>All queries in this group</strong></p>
<ul>
<li><a href="/hebrew/text?iid=7334&amp;page=1&amp;mr=r&amp;qw=q">All occurrences of טף</a></li>
<li><a href="/hebrew/text?iid=7333&amp;page=1&amp;mr=r&amp;qw=q">Co-occurrences of טף with any of Deut 31:12 group</a></li>
<li><a href="/hebrew/text?iid=7328&amp;page=1&amp;mr=r&amp;qw=q">Co-occurrences of טף with all members of Deut 31:12 group</a></li>
<li><a href="/hebrew/text?iid=7329&amp;page=1&amp;mr=r&amp;qw=q">Deut 31:12 audience without the טף ('litle ones')</a></li>
</ul>

**MQL stored with the BHSA 2021 execution**

...

**MQL stored with the BHSA 2021 execution**

SELECT ALL OBJECTS WHERE
[verse 
  [UNORDEREDGROUP
    [word FOCUS lex = "VP/"]
    [word FOCUS lex = "GR/"]
  ]
]

**First rows of the control table**

,shebanq_query_id,shebanq_short_name,shebanq_query_name,bhsa_version,shebanq_result_count,shebanq_csv_row_count,shebanq_reference,reference,book,chapter,verse,slot,text,ktv,lexeme-t,part-of-speech,gender,phrase#,tab
0,7335,ger_with_taph,Where occur טף and גר together,2021,6,6,Deuteronomium 29:10,Deuteronomy 29:10,Deuteronomium,29,10,109717,טַפְּכֶ֣ם,NaN,VP/,subs,m,1,9
1,7335,ger_with_taph,Where occur טף and גר together,2021,6,6,Deuteronomium 29:10,Deuteronomy 29:10,Deuteronomium,29,10,109720,גֵ֣רְךָ֔,NaN,GR/,subs,m,1,9
2,7335,ger_with_taph,Where occur טף and גר together,2021,6,6,Deuteronomium 31:12,Deuteronomy 31:12,Deuteronomium,31,12,110931,טַּ֔ף,NaN,VP/,subs,m,2,4
3,7335,ger_with_taph,Where occur טף and גר together,2021,6,6,Deuteronomium 31:12,Deuteronomy 31:12,Deuteronomium,31,12,110933,גֵרְךָ֖,NaN,GR/,subs,m,2,4
4,7335,ger_with_taph,Where occur טף and גר together,2021,6,6,Josua 8:35,Joshua 8:35,Josua,8,35,117735,טַּ֔ף,NaN,VP/,subs,m,5,4


,query_id,short_name,query_name,is_shared,is_published,result_count,csv_row_count,status
0,7334,all_taph,All occurrence of טף,True,False,42,42,ok
1,7333,taph_with_any_audience_member,Co-occurrence of טף with any of Deut 31:12 group,True,False,65,65,ok
2,7328,taph_with_complete_audience,Co-occurrence of טף with all members of Deut 3...,True,False,4,4,ok
3,7329,audience_without_taph,Deut 31:12 audience without the טף ('litle ones'),True,False,0,0,ok
4,7335,ger_with_taph,Where occur טף and גר together,True,False,6,6,ok


# 7 - Export the recorded query data<a class="anchor" id="bullet7"></a>
##### [Back to ToC](#TOC)

The CSV files make the rows easy to inspect. The JSON export also retains multiline descriptions and the MQL query text.


In [7]:
# Export the recorded query data
DETAIL_CSV = OUTPUT_DIR / "query_details.csv"
DETAIL_JSON = OUTPUT_DIR / "shebanq_query_details.json"
COMBINED_CSV = OUTPUT_DIR / "controls.csv"

query_details.to_csv(DETAIL_CSV, index=False, encoding="utf-8")

detail_manifest = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "base_url": BASE_URL,
    "bhsa_version": BHSA_VERSION,
    "authentication_used": False,
    "access_note": (
        "The records are shared and anonymously readable. Formal SHEBANQ "
        "publication status is recorded separately for each BHSA execution."
    ),
    "source_catalogue": QUERY_CATALOGUE.relative_to(REPOSITORY_DIR).as_posix(),
    "book_name_mapping": BOOK_NAME_MAP,
    "queries": query_details.where(pd.notna(query_details), None).to_dict("records"),
}
DETAIL_JSON.write_text(
    json.dumps(detail_manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

combined_control_results = (
    pd.concat(control_frames.values(), ignore_index=True, sort=False)
    if control_frames
    else pd.DataFrame()
)
combined_control_results.to_csv(COMBINED_CSV, index=False, encoding="utf-8")

print(f"Detailed CSV: {DETAIL_CSV.relative_to(REPOSITORY_DIR).as_posix()}")
print(f"Detailed JSON: {DETAIL_JSON.relative_to(REPOSITORY_DIR).as_posix()}")
print(f"Combined control: {COMBINED_CSV.relative_to(REPOSITORY_DIR).as_posix()}")


Detailed CSV: SHEBANQ-BHSA/results/public_query_control/query_details.csv
Detailed JSON: SHEBANQ-BHSA/results/public_query_control/shebanq_query_details.json
Combined control: SHEBANQ-BHSA/results/public_query_control/controls.csv


# 8 - Compare SHEBANQ query 7334 with Context-Fabric<a class="anchor" id="bullet8"></a>
##### [Back to ToC](#TOC)

[SHEBANQ query 7334](https://shebanq.ancient-data.org/hebrew/text?iid=7334&page=1&mr=r&qw=q) is the positive control for all occurrences of `טף`. We join its focus rows to `context_fabric/results/taf_occurrences.csv` by the BHSA word slot and then compare the mapped reference. This checks both identity and location without treating the researcher-supplied contextual category as a SHEBANQ feature.


In [8]:
# Compare query 7334 with Context Fabric
if not CONTEXT_FABRIC_INPUT.exists():
    raise FileNotFoundError(
        "Run the Context Fabric notebook first; its 42-row result is the primary input."
    )

context_rows = pd.read_csv(CONTEXT_FABRIC_INPUT)
shebanq_rows = control_frames["all_taph"].copy()

shebanq_control = shebanq_rows[["slot", "book", "shebanq_reference", "reference"]].rename(
    columns={"book": "shebanq_book"}
)
context_control = context_rows[["word_node", "reference"]].rename(
    columns={
        "word_node": "slot",
        "reference": "context_fabric_reference",
    }
)

comparison = shebanq_control.merge(
    context_control,
    on="slot",
    how="outer",
    indicator="row_source",
)
comparison["reference_agrees"] = (
    comparison["reference"] == comparison["context_fabric_reference"]
)
comparison = comparison.sort_values("slot", na_position="last").reset_index(drop=True)
COMPARISON_CSV = OUTPUT_DIR / "bhsa_comparison.csv"
comparison.to_csv(COMPARISON_CSV, index=False, encoding="utf-8")

control_summary = pd.DataFrame(
    [
        {
            "query_id": detail["query_id"],
            "short_name": detail["short_name"],
            "result_count": detail.get("result_count"),
            "focus_rows": detail.get("csv_row_count"),
            "unique_references": control_frames[detail["short_name"]]["reference"].nunique(),
        }
        for detail in detail_rows
        if detail.get("status") == "ok"
    ]
)

display(control_summary)
display(comparison.head(10))
print(f"Comparison rows: {len(comparison)}")
print(f"Rows present in both routes: {(comparison['row_source'] == 'both').sum()}")
print(f"Mapped references agreeing: {comparison['reference_agrees'].sum()}")


,query_id,short_name,result_count,focus_rows,unique_references
0,7334,all_taph,42,42,42
1,7333,taph_with_any_audience_member,65,65,27
2,7328,taph_with_complete_audience,4,4,1
3,7329,audience_without_taph,0,0,0
4,7335,ger_with_taph,6,6,3


,slot,shebanq_book,shebanq_reference,reference,context_fabric_reference,row_source,reference_agrees
0,18987,Genesis,Genesis 34:29,Genesis 34:29,Genesis 34:29,both,True
1,24293,Genesis,Genesis 43:8,Genesis 43:8,Genesis 43:8,both,True
2,25818,Genesis,Genesis 45:19,Genesis 45:19,Genesis 45:19,both,True
3,26076,Genesis,Genesis 46:5,Genesis 46:5,Genesis 46:5,both,True
4,26806,Genesis,Genesis 47:12,Genesis 47:12,Genesis 47:12,both,True
5,27147,Genesis,Genesis 47:24,Genesis 47:24,Genesis 47:24,both,True
6,28400,Genesis,Genesis 50:8,Genesis 50:8,Genesis 50:8,both,True
7,28657,Genesis,Genesis 50:21,Genesis 50:21,Genesis 50:21,both,True
8,34034,Exodus,Exodus 10:10,Exodus 10:10,Exodus 10:10,both,True
9,34363,Exodus,Exodus 10:24,Exodus 10:24,Exodus 10:24,both,True


Comparison rows: 42
Rows present in both routes: 42
Mapped references agreeing: 42


# 9 - Final checks<a class="anchor" id="bullet9"></a>
##### [Back to ToC](#TOC)

The audit checks anonymous access, the shared/status metadata, one result table for each of the five queries, the positive 42-row control, the zero-row negative control, and exact reference-and-slot agreement with Context-Fabric.


In [9]:
# Final checks
assert len(query_details) == len(validated_specs)
assert set(query_details["query_id"]) == set(query_ids)
assert (query_details["status"] == "ok").all(), query_details.loc[
    query_details["status"] != "ok", ["query_id", "error"]
]
assert query_details["publicly_readable_without_login"].all()
assert query_details["is_shared"].all()
assert (query_details["execution_status"] == "good").all()
assert query_details["mql"].fillna("").str.strip().ne("").all()
assert len(control_frames["all_taph"]) == 42
assert len(control_frames["audience_without_taph"]) == 0
assert len(comparison) == 42
assert (comparison["row_source"] == "both").all()
assert comparison["reference_agrees"].all()

audit = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "bhsa_version": BHSA_VERSION,
    "authentication_used": False,
    "query_ids": query_ids,
    "all_queries_shared": bool(query_details["is_shared"].all()),
    "formal_publication_flags": {
        str(row.query_id): bool(row.is_published)
        for row in query_details.itertuples()
    },
    "query_result_counts": {
        str(row.query_id): int(row.result_count)
        for row in query_details.itertuples()
    },
    "query_focus_rows": {
        str(row.query_id): int(row.csv_row_count)
        for row in query_details.itertuples()
    },
    "book_name_mapping": BOOK_NAME_MAP,
    "context_fabric_input": CONTEXT_FABRIC_INPUT.relative_to(REPOSITORY_DIR).as_posix(),
    "all_taph_control_rows": len(comparison),
    "slots_present_in_both_routes": int((comparison["row_source"] == "both").sum()),
    "mapped_references_agreeing": int(comparison["reference_agrees"].sum()),
    "exact_context_fabric_control_match": bool(
        (comparison["row_source"] == "both").all()
        and comparison["reference_agrees"].all()
    ),
}
(OUTPUT_DIR / "control_audit.json").write_text(
    json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8"
)

print("All five anonymous SHEBANQ controls passed.")
print("Query 7334 agrees exactly with the 42 Context Fabric word slots and references.")


All five anonymous SHEBANQ controls passed.
Query 7334 agrees exactly with the 42 Context Fabric word slots and references.


# 10 - Reading the control results<a class="anchor" id="bullet10"></a>
##### [Back to ToC](#TOC)

SHEBANQ's result and focus-row counts depend on the MQL focus objects. They must therefore be read together with the returned MQL rather than treated as interchangeable measures.

The principal result here is simple: [SHEBANQ query 7334](https://shebanq.ancient-data.org/hebrew/text?iid=7334&page=1&mr=r&qw=q) and Context-Fabric should identify the same 42 BHSA word slots after the explicitly documented book-name mapping.

The remaining queries test broader, complete, and negative audience configurations. They are just some controls on the corpus investigation.


# 11 - Required libraries<a class="anchor" id="bullet11"></a>
##### [Back to ToC](#TOC)

The notebook uses `requests`, `pandas`, and Jupyter/IPython display helpers. Network access is required for the public SHEBANQ calls. No credentials are read from a file or environment variable.


# 12 - Input and output overview<a class="anchor" id="bullet12"></a>
##### [Back to ToC](#TOC)

**Maintained input**

* `SHEBANQ-BHSA/public_shebanq_queries.json`
* `context_fabric/results/taf_occurrences.csv`

**Generated output**

* `SHEBANQ-BHSA/results/public_query_control/query_details.csv`
* `SHEBANQ-BHSA/results/public_query_control/shebanq_query_details.json`
* `SHEBANQ-BHSA/results/public_query_control/controls.csv`
* `SHEBANQ-BHSA/results/public_query_control/query_results/ (five CSVs named in the export mapping)`
* `SHEBANQ-BHSA/results/public_query_control/bhsa_comparison.csv`
* `SHEBANQ-BHSA/results/public_query_control/control_audit.json`

These recorded outputs are tracked with the notebook. Local environments, caches, corpus files, and authentication material remain ignored.


# 13 - Responsible disclosure<a class="anchor" id="bullet13"></a>
##### [Back to ToC](#TOC)

The author used OpenAI Codex to assist with Python code and data analysis. The author reviewed and edited the material and takes responsibility for the content and interpretation.

# 14 - Licences<a class="anchor" id="bullet14"></a>
##### [Back to ToC](#TOC)

The original code in this notebook is licensed under the [MIT License](../LICENSE).
The original research writing and explanatory text are licensed under [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/).
Credit Tony Jurg, link the licence and repository, and indicate any changes when reusing the original text; see the [text licence](../LICENSE-TEXT.md).

Quoted source material, retrieved passages, and datasets retain their own terms.
See [Sources, attribution, and reuse](../DATA_RIGHTS.md) for their licences and attribution requirements.


# 15 - Notebook details<a class="anchor" id="bullet15"></a>
##### [Back to ToC](#TOC)

<div style="float: left;">
  <table>
    <tr><td><strong>Author</strong></td><td>Tony Jurg</td></tr>
    <tr><td><strong>Version</strong></td><td>1.6</td></tr>
    <tr><td><strong>Date</strong></td><td>October 26, 2026</td></tr>
  </table>
</div>